# Notebook 03 - Agentes Especializados Por Tools

Cada especialista solo tiene acceso a las tools que necesita:
- Especialista de conocimiento: solo `tool_search_knowledge` (AI Search).
- Especialista de cliente: solo `tool_buscar_cliente` y `tool_listar_productos`.
- Especialista de soporte: solo `tool_consultar_tickets`.

In [60]:
import json
import time
import subprocess
import tempfile
from urllib.parse import urlparse
import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

with open('workshop_config.json', 'r') as f:
    cfg = json.load(f)

assert cfg.get('ai_search_ready'), 'Ejecuta primero notebook 01'
assert cfg.get('function_app_ready'), 'Ejecuta primero notebook 02'
assert cfg.get('foundry_project_endpoint'), 'Falta foundry_project_endpoint en workshop_config.json'

RESPONSES_API_VERSION = '2025-03-01-preview'
FOUNDRY_AGENTS_API_VERSION = cfg.get('foundry_agents_api_version', '2025-05-15-preview')
FOUNDRY_PROJECT_CONNECTIONS_API_VERSION = '2025-06-01'

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, 'https://cognitiveservices.azure.com/.default')
client = AzureOpenAI(
    azure_endpoint=cfg['azure_openai_endpoint'],
    azure_ad_token_provider=token_provider,
    api_version=RESPONSES_API_VERSION
)

MODEL = cfg['model_deployment_name']
PREFIX = cfg['resource_prefix']
FOUNDRY_WORKSPACE_NAME = cfg.get('foundry_workspace_name', '')
FOUNDRY_WORKSPACE_RG = cfg.get('foundry_workspace_resource_group', '')
FOUNDRY_ACCOUNT_RESOURCE_GROUP = cfg.get('foundry_account_resource_group', '')
FOUNDRY_SUBSCRIPTION_ID = cfg.get('azure_subscription_id', '')
SEARCH_ENDPOINT = cfg['ai_search']['endpoint']
SEARCH_INDEX = cfg['ai_search']['index_name']
SEARCH_API_VERSION = cfg['ai_search']['api_version']
SEARCH_ADMIN_KEY = cfg['ai_search']['admin_key']
USE_VECTOR_SEARCH = cfg['ai_search'].get('use_vector_search', False)
EMBEDDING_DEPLOYMENT = cfg['ai_search'].get('embedding_model_deployment') or cfg.get('embedding_model_deployment', '')
FUNCTION_BASE_URL = cfg['function_app']['base_url']
FOUNDRY_PROJECT_ENDPOINT = cfg['foundry_project_endpoint'].rstrip('/')
FOUNDRY_PROJECT_NAME = FOUNDRY_PROJECT_ENDPOINT.split('/api/projects/', 1)[1].split('?', 1)[0]
FOUNDRY_ACCOUNT_NAME = urlparse(cfg['azure_openai_endpoint']).hostname.split('.')[0]
FOUNDRY_AI_SEARCH_CONNECTION = cfg.get('foundry_connections', {}).get('ai_search', f'{PREFIX}-ai-search')
FOUNDRY_FUNCTION_CONNECTION = cfg.get('foundry_connections', {}).get('function_api', f'{PREFIX}-function-api')

print('OK contexto de especialistas')
print('Responses API version:', RESPONSES_API_VERSION)
print('Vector search habilitado:', USE_VECTOR_SEARCH)
print('Foundry endpoint:', FOUNDRY_PROJECT_ENDPOINT)

OK contexto de especialistas
Responses API version: 2025-03-01-preview
Vector search habilitado: True
Foundry endpoint: https://ai-williamta-9386.services.ai.azure.com/api/projects/ai-williamta-9386-project


In [61]:
def _foundry_headers() -> dict:
    token = credential.get_token('https://ai.azure.com/.default').token
    return {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

def _foundry_list_agents() -> dict:
    r = requests.get(
        f"{FOUNDRY_PROJECT_ENDPOINT}/agents?api-version={FOUNDRY_AGENTS_API_VERSION}",
        headers=_foundry_headers(),
        timeout=30
    )
    if r.status_code != 200:
        raise RuntimeError(f'No se pudo listar agentes Foundry: {r.status_code} {r.text}')
    return {a['id']: a for a in r.json().get('data', [])}

def _delete_foundry_agent(agent_name: str) -> None:
    requests.delete(
        f"{FOUNDRY_PROJECT_ENDPOINT}/agents/{agent_name}?api-version={FOUNDRY_AGENTS_API_VERSION}",
        headers=_foundry_headers(),
        timeout=30
    )

def _run_az(args: list[str]) -> dict:
    cmd = ['az', *args, '-o', 'json']
    p = subprocess.run(cmd, check=True, capture_output=True, text=True)
    out = (p.stdout or '').strip()
    return json.loads(out) if out else {}

def _resolve_foundry_workspace() -> tuple[str, str]:
    if FOUNDRY_WORKSPACE_NAME and FOUNDRY_WORKSPACE_RG:
        return FOUNDRY_WORKSPACE_NAME, FOUNDRY_WORKSPACE_RG
    workspaces = _run_az(['ml', 'workspace', 'list'])
    if not isinstance(workspaces, list) or not workspaces:
        raise RuntimeError('No se encontraron workspaces de Azure ML para crear conexiones.')
    preferred = [w for w in workspaces if str(w.get('name', '')).startswith('aiproj-')]
    ws = preferred[0] if preferred else workspaces[0]
    return ws['name'], ws['resourceGroup']

def _resolve_foundry_account_resource_group() -> str:
    if FOUNDRY_ACCOUNT_RESOURCE_GROUP:
        return FOUNDRY_ACCOUNT_RESOURCE_GROUP
    accounts = _run_az(['cognitiveservices', 'account', 'list'])
    if not isinstance(accounts, list):
        raise RuntimeError('No se pudo listar cuentas Cognitive Services para resolver el RG del proyecto Foundry.')
    for a in accounts:
        if a.get('name') == FOUNDRY_ACCOUNT_NAME:
            rg = a.get('resourceGroup', '')
            if rg:
                return rg
    raise RuntimeError(f'No se encontro la cuenta Foundry {FOUNDRY_ACCOUNT_NAME} en la suscripcion actual.')

def _build_foundry_connection_id(resource_group: str, connection_name: str) -> str:
    if not FOUNDRY_SUBSCRIPTION_ID:
        raise RuntimeError('Falta azure_subscription_id para construir project_connection_id.')
    return (
        f"/subscriptions/{FOUNDRY_SUBSCRIPTION_ID}"
        f"/resourceGroups/{resource_group}"
        f"/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_ACCOUNT_NAME}"
        f"/projects/{FOUNDRY_PROJECT_NAME}/connections/{connection_name}"
    )

def _project_connection_url(resource_group: str, connection_name: str) -> str:
    return (
        f"https://management.azure.com/subscriptions/{FOUNDRY_SUBSCRIPTION_ID}"
        f"/resourceGroups/{resource_group}"
        f"/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_ACCOUNT_NAME}"
        f"/projects/{FOUNDRY_PROJECT_NAME}/connections/{connection_name}"
        f"?api-version={FOUNDRY_PROJECT_CONNECTIONS_API_VERSION}"
    )

def ensure_workspace_connection(name: str, conn_type: str, target: str, credentials: dict, endpoint: str | None = None) -> dict:
    account_rg = _resolve_foundry_account_resource_group()
    foundry_conn_id = _build_foundry_connection_id(account_rg, name)
    url = _project_connection_url(account_rg, name)
    category = 'CognitiveSearch' if conn_type == 'azure_ai_search' else 'GenericRest'
    auth_type = 'ApiKey' if credentials.get('type') == 'api_key' else 'None'
    try:
        existing = _run_az(['rest', '--method', 'get', '--url', url])
        ex_props = existing.get('properties', {}) if isinstance(existing, dict) else {}
        if ex_props.get('target') == target and ex_props.get('authType') == auth_type:
            return {'name': name, 'workspace': '', 'resource_group': account_rg, 'workspace_connection_id': '', 'connection_id': foundry_conn_id, 'created': False}
    except Exception:
        pass

    body = {'properties': {'category': category, 'authType': auth_type, 'target': target}}
    if auth_type == 'ApiKey':
        body['properties']['credentials'] = {'key': credentials.get('key', '')}
    _run_az(['rest', '--method', 'put', '--url', url, '--body', json.dumps(body), '--headers', 'Content-Type=application/json'])
    return {'name': name, 'workspace': '', 'resource_group': account_rg, 'workspace_connection_id': '', 'connection_id': foundry_conn_id, 'created': True}

def ensure_foundry_prompt_agent(agent_name: str, instructions: str, tools: list | None = None, metadata: dict | None = None) -> dict:
    tools = tools or []
    metadata = metadata or {}
    existing = _foundry_list_agents()
    if agent_name in existing:
        latest = existing[agent_name].get('versions', {}).get('latest', {})
        existing_instructions = latest.get('definition', {}).get('instructions', '')
        existing_tools = latest.get('definition', {}).get('tools', [])
        existing_meta = latest.get('metadata', {})
        tools_equal = json.dumps(existing_tools, sort_keys=True) == json.dumps(tools, sort_keys=True)
        instructions_equal = existing_instructions == instructions
        metadata_equal = json.dumps(existing_meta, sort_keys=True) == json.dumps(metadata, sort_keys=True)
        if tools_equal and instructions_equal and metadata_equal:
            return {'id': agent_name, 'name': existing[agent_name].get('name', agent_name), 'created': False}
        _delete_foundry_agent(agent_name)

    payload = {
        'name': agent_name,
        'metadata': metadata,
        'definition': {
            'kind': 'prompt',
            'model': MODEL,
            'instructions': instructions,
            'tools': tools
        }
    }
    r = requests.post(
        f"{FOUNDRY_PROJECT_ENDPOINT}/agents?api-version={FOUNDRY_AGENTS_API_VERSION}",
        headers=_foundry_headers(),
        json=payload,
        timeout=30
    )
    if r.status_code not in (200, 201):
        raise RuntimeError(f'No se pudo crear agente Foundry {agent_name}: {r.status_code} {r.text}')
    data = r.json()
    return {'id': data['id'], 'name': data.get('name', data['id']), 'created': True}

def search_knowledge(query: str) -> str:
    url = f"{SEARCH_ENDPOINT}/indexes/{SEARCH_INDEX}/docs/search?api-version={SEARCH_API_VERSION}"
    body = {'top': 3, 'select': 'source,content'}
    if USE_VECTOR_SEARCH and EMBEDDING_DEPLOYMENT:
        qv = client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=query).data[0].embedding
        body['vectorQueries'] = [{
            'kind': 'vector',
            'vector': qv,
            'fields': 'contentVector',
            'k': 3
        }]
    else:
        body['search'] = query

    r = requests.post(url, headers={'Content-Type': 'application/json', 'api-key': SEARCH_ADMIN_KEY}, json=body, timeout=30)
    if r.status_code != 200:
        return json.dumps({'error': r.text})
    return json.dumps(r.json().get('value', []), ensure_ascii=False)

def call_business_function(function_name: str, payload: dict) -> str:
    r = requests.post(f"{FUNCTION_BASE_URL}/{function_name}", json=payload, timeout=30)
    return r.text

In [41]:
tools_conocimiento = [
  {
    'type': 'function',
    'name': 'tool_search_knowledge',
    'description': 'Busca conocimiento en AI Search',
    'parameters': {
      'type': 'object',
      'properties': {'query': {'type': 'string'}},
      'required': ['query']
    }
  }
]

tools_cliente = [
  {
    'type': 'function',
    'name': 'tool_buscar_cliente',
    'description': 'Busca cliente por cedula',
    'parameters': {
      'type': 'object',
      'properties': {'cedula': {'type': 'string'}},
      'required': ['cedula']
    }
  },
  {
    'type': 'function',
    'name': 'tool_listar_productos',
    'description': 'Lista productos disponibles',
    'parameters': {
      'type': 'object',
      'properties': {},
      'required': []
    }
  }
]

tools_soporte = [
  {
    'type': 'function',
    'name': 'tool_consultar_tickets',
    'description': 'Consulta tickets de un cliente por cliente_id',
    'parameters': {
      'type': 'object',
      'properties': {'cliente_id': {'type': 'string'}},
      'required': ['cliente_id']
    }
  }
]

ESPECIALISTAS = {
    'conocimiento': {
        'name': f"{PREFIX}-esp-conocimiento",
        'instructions': 'Especialista de conocimiento. Debes llamar tool_search_knowledge en cada consulta y responder solo con datos devueltos por esa tool. Si el resultado incluye horarios o canales, listalos explicitamente.',
        'tools': tools_conocimiento
    },
    'cliente': {
        'name': f"{PREFIX}-esp-cliente",
        'instructions': 'Especialista de cliente. Debes usar solo tool_buscar_cliente o tool_listar_productos y responder con la data obtenida, sin inventar.',
        'tools': tools_cliente
    },
    'soporte': {
        'name': f"{PREFIX}-esp-soporte",
        'instructions': 'Especialista de soporte. Debes usar tool_consultar_tickets y responder con el resultado exacto.',
        'tools': tools_soporte
    }
}

print('Especialistas listos (Responses API):')
for k, v in ESPECIALISTAS.items():
    print(k, '->', v['name'])

Especialistas listos (Responses API):
conocimiento -> contoso-user01-esp-conocimiento
cliente -> contoso-user01-esp-cliente
soporte -> contoso-user01-esp-soporte


In [29]:
def _tool_dispatch(fn: str, args: dict) -> str:
    if fn == 'tool_search_knowledge':
        return search_knowledge(args['query'])
    if fn == 'tool_buscar_cliente':
        return call_business_function('buscar_cliente', {'cedula': args['cedula']})
    if fn == 'tool_listar_productos':
        return call_business_function('listar_productos_disponibles', {})
    if fn == 'tool_consultar_tickets':
        return call_business_function('consultar_tickets_cliente', {'cliente_id': args['cliente_id']})
    return json.dumps({'error': f'tool no permitida: {fn}'})

def _run_response_agent(instructions: str, tools: list, pregunta: str) -> str:
    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=pregunta,
        tools=tools,
        tool_choice='required'
    )

    while True:
        function_calls = [o for o in response.output if getattr(o, 'type', '') == 'function_call']
        if not function_calls:
            return response.output_text or ''

        tool_outputs = []
        for call in function_calls:
            args = json.loads(call.arguments or '{}')
            output = _tool_dispatch(call.name, args)
            tool_outputs.append({
                'type': 'function_call_output',
                'call_id': call.call_id,
                'output': output
            })

        response = client.responses.create(
            model=MODEL,
            previous_response_id=response.id,
            input=tool_outputs
        )

def ejecutar_especialista(tipo: str, pregunta: str) -> str:
    if tipo not in ESPECIALISTAS:
        return 'Especialista no soportado'
    esp = ESPECIALISTAS[tipo]
    return _run_response_agent(esp['instructions'], esp['tools'], pregunta)

print('Motor de especialistas listo (Responses API)')

Motor de especialistas listo (Responses API)


In [62]:
current_foundry = cfg.get('foundry_agents', {})

target_names = {
    'conocimiento': current_foundry.get('conocimiento', {}).get('name', f"{PREFIX}-esp-conocimiento-foundry"),
    'cliente': current_foundry.get('cliente', {}).get('name', f"{PREFIX}-esp-cliente-foundry"),
    'soporte': current_foundry.get('soporte', {}).get('name', f"{PREFIX}-esp-soporte-foundry")
}

ai_conn = ensure_workspace_connection(
    name=FOUNDRY_AI_SEARCH_CONNECTION,
    conn_type='azure_ai_search',
    target=SEARCH_ENDPOINT,
    endpoint=SEARCH_ENDPOINT,
    credentials={'type': 'api_key', 'key': SEARCH_ADMIN_KEY}
)
fn_conn = ensure_workspace_connection(
    name=FOUNDRY_FUNCTION_CONNECTION,
    conn_type='custom',
    target=FUNCTION_BASE_URL,
    credentials={'type': 'none'}
)
print(f"Conexion AI Search: {ai_conn['name']} ({'creada' if ai_conn['created'] else 'existente'})")
print(f"Conexion Function API: {fn_conn['name']} ({'creada' if fn_conn['created'] else 'existente'})")
assert ai_conn.get('connection_id'), 'No se pudo resolver el ID de la conexion AI Search'
assert fn_conn.get('connection_id'), 'No se pudo resolver el ID de la conexion Function API'
AI_SEARCH_CONNECTION_ID = ai_conn['connection_id']
FUNCTION_CONNECTION_ID = fn_conn['connection_id']
cfg['foundry_workspace_name'] = ai_conn['workspace']
cfg['foundry_workspace_resource_group'] = ai_conn['resource_group']
cfg['foundry_account_resource_group'] = ai_conn['resource_group']

foundry_specs = {
    'conocimiento': {
        'instructions': 'Especialista de conocimiento. Debes usar la tool ai_search_kb en cada consulta y responder solo con datos recuperados del indice. Si no hay resultados, dilo explicitamente.',
        'tools': [
            {
                'type': 'azure_ai_search',
                'name': 'ai_search_kb',
                'description': 'Consulta el indice de conocimiento para responder FAQ, horarios y politicas.',
                'azure_ai_search': {
                    'indexes': [
                        {
                            'project_connection_id': AI_SEARCH_CONNECTION_ID,
                            'index_name': SEARCH_INDEX,
                            'query_type': 'simple',
                            'top_k': 5,
                            'endpoint': SEARCH_ENDPOINT,
                            'authentication': {
                                'type': 'api_key',
                                'key': SEARCH_ADMIN_KEY
                            }
                        }
                    ],
                    'indexConfiguration': {
                        'projectConnectionId': AI_SEARCH_CONNECTION_ID,
                        'indexName': SEARCH_INDEX
                    }
                }
            }
        ],
        'metadata': {
            'tool_mode': 'native',
            'data_source_type': 'azure_ai_search',
            'ai_search_endpoint': SEARCH_ENDPOINT,
            'ai_search_index': SEARCH_INDEX
        }
    },
    'cliente': {
        'instructions': 'Especialista de cliente. Usa la tool business_api_cliente para consultar por cedula o listar productos y responde solo con la data retornada.',
        'tools': [
            {
                'type': 'openapi',
                'name': 'business_api_cliente',
                'description': 'API de cliente para buscar por cedula y listar productos disponibles.',
                'openapi': {
                    'name': 'business_api_cliente',
                    'spec': {
                        'openapi': '3.0.1',
                        'info': {'title': 'Contoso Cliente API', 'version': '1.0.0'},
                        'servers': [{'url': FUNCTION_BASE_URL}],
                        'paths': {
                            '/buscar_cliente': {
                                'post': {
                                    'operationId': 'buscar_cliente',
                                    'description': 'Busca cliente por cedula',
                                    'requestBody': {
                                        'required': True,
                                        'content': {
                                            'application/json': {
                                                'schema': {
                                                    'type': 'object',
                                                    'properties': {'cedula': {'type': 'string'}},
                                                    'required': ['cedula']
                                                }
                                            }
                                        }
                                    },
                                    'responses': {'200': {'description': 'OK'}}
                                }
                            },
                            '/listar_productos_disponibles': {
                                'post': {
                                    'operationId': 'listar_productos_disponibles',
                                    'description': 'Lista productos disponibles',
                                    'responses': {'200': {'description': 'OK'}}
                                }
                            }
                        }
                    },
                    'auth': {'type': 'connection', 'connection_id': FUNCTION_CONNECTION_ID}
                }
            }
        ],
        'metadata': {
            'tool_mode': 'native',
            'business_api_base_url': FUNCTION_BASE_URL
        }
    },
    'soporte': {
        'instructions': 'Especialista de soporte. Usa la tool business_api_soporte para consultar tickets y responde unicamente con ese resultado.',
        'tools': [
            {
                'type': 'openapi',
                'name': 'business_api_soporte',
                'description': 'API de soporte para consultar tickets de un cliente.',
                'openapi': {
                    'name': 'business_api_soporte',
                    'spec': {
                        'openapi': '3.0.1',
                        'info': {'title': 'Contoso Soporte API', 'version': '1.0.0'},
                        'servers': [{'url': FUNCTION_BASE_URL}],
                        'paths': {
                            '/consultar_tickets_cliente': {
                                'post': {
                                    'operationId': 'consultar_tickets_cliente',
                                    'description': 'Consulta tickets por cliente_id',
                                    'requestBody': {
                                        'required': True,
                                        'content': {
                                            'application/json': {
                                                'schema': {
                                                    'type': 'object',
                                                    'properties': {'cliente_id': {'type': 'string'}},
                                                    'required': ['cliente_id']
                                                }
                                            }
                                        }
                                    },
                                    'responses': {'200': {'description': 'OK'}}
                                }
                            }
                        }
                    },
                    'auth': {'type': 'connection', 'connection_id': FUNCTION_CONNECTION_ID}
                }
            }
        ],
        'metadata': {
            'tool_mode': 'native',
            'business_api_base_url': FUNCTION_BASE_URL
        }
    }
}

foundry_agents = {}
for tipo in ('conocimiento', 'cliente', 'soporte'):
    spec = foundry_specs[tipo]
    result = ensure_foundry_prompt_agent(
        target_names[tipo],
        spec.get('instructions', ESPECIALISTAS[tipo]['instructions']),
        tools=spec['tools'],
        metadata=spec['metadata']
    )
    foundry_agents[tipo] = {'id': result['id'], 'name': result['name']}
    estado = 'creado/actualizado' if result['created'] else 'ya existia'
    print(f"Foundry agente {tipo}: {foundry_agents[tipo]['name']} ({estado})")

cfg['agentes_especializados'] = {
    k: {'name': v['name']} for k, v in ESPECIALISTAS.items()
}
cfg['foundry_agents'] = {**cfg.get('foundry_agents', {}), **foundry_agents}
cfg['foundry_connections'] = {
    'ai_search': FOUNDRY_AI_SEARCH_CONNECTION,
    'ai_search_id': AI_SEARCH_CONNECTION_ID,
    'function_api': FOUNDRY_FUNCTION_CONNECTION,
    'function_api_id': FUNCTION_CONNECTION_ID
}
cfg['agentes_runtime'] = 'responses_api'
cfg['especialistas_ready'] = True
with open('workshop_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)
print('OK especialistas guardados en workshop_config.json')

Conexion AI Search: user01-ai-search (existente)
Conexion Function API: user01-function-api (existente)
Foundry agente conocimiento: contoso-user01-esp-conocimiento-foundry (creado/actualizado)
Foundry agente cliente: contoso-user01-esp-cliente-foundry (ya existia)
Foundry agente soporte: contoso-user01-esp-soporte-foundry (ya existia)
OK especialistas guardados en workshop_config.json
